In [1]:

# HOW TO CREATE CONDA ENV
"""conda create -n dl python=3.10
conda activate dl

pip install torch torchvision medmnist
pip install grad-cam
pip install seaborn"""


# IF YOU USE GOOGLE COLAB

"""!pip install python==3.10
!pip install medmnist
!pip install grad-cam
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
!
!pip install pillow==11.1.0

from google.colab import drive
drive.mount('/content/drive')

import sys
import os
os.chdir('/content/drive/MyDrive/DL/')"""


# Libraries required: torch, seaborn, medmnist, grad-cam

"!pip install python==3.10\n!pip install medmnist\n!pip install grad-cam\n!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126\n!\n!pip install pillow==11.1.0\n\nfrom google.colab import drive\ndrive.mount('/content/drive')\n\nimport sys\nimport os\nos.chdir('/content/drive/MyDrive/DL/')"

In [1]:
# import libraries 

import importlib
import src.train_utils
import src.utils    
import torch    # PyTorch
import os

#importlib.reload(utils)
#importlib.reload(train_utils)

# CHECK IF TORCH IS INSTALLED

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "CPU")

ModuleNotFoundError: No module named 'src'

## 1. BINARY CLASSIFICATION

### 1.1 Data loading + partitioning into datasets (training, validation, test)

Examine the outcomes of the code, see what the sample data looks like.

In [ ]:
from src.utils import binary_classification_dataset, binary_classification_training, binary_classification_validate, binary_classification_GRADCAM

# DO NOT CHANGE

if_train = False            # true, jeśli uczenie na żywo

device = "cuda" if torch.cuda.is_available() else "cpu"
info, n_classes, train_bin, val_bin, test_bin, train_loader_bin, val_loader_bin, test_loader_bin = binary_classification_dataset(device)

### 1.2 Model training and validation

TODO: Select hyperparameters (number of epoch and optimizer) to achieve (in your opinion) the best possible performance of the CNN model for binary classification. Consider the training process, quality metrics, and the GRAD-CAM activation map.


**JUSTIFY YOUR CHOICES.**

`Number of epochs to choose: 5, 10, 20, 30, 40, 100`

`Optimizers to choose: Adam (lr = 0.001) lub SGD (lr = 0.01)`

`Each time, go through the steps in order:`

`*select hyperparameters in 1.2A* -> *1.2B* -> *1.2C* -> *adjust hyperparameters in 1.2A* -> *1.2B* ....`

---
1.2A CNN architecture + hyperparameters + training

In [ ]:
from src.utils import binary_classification_dataset, binary_classification_training, binary_classification_validate, binary_classification_GRADCAM
from src.models_architecture import BinaryCNN
import torch.optim as optim
import torch.nn as nn

############################################################################
######                    model's architecture                        ######
############################################################################

class BinaryModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), # convolutional layer, parameters: number of channels it accepts; number of filters; kernel size
            nn.ReLU(),  # activation layer (ReLU)
            nn.MaxPool2d(2),    # max pooling layer (params: kernel size)
            nn.Conv2d(32,64,3,padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((7,7)) # avg pooling layer (params: kernel size)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),   # layer that creates vector
            nn.Linear(64*7*7,128), # fully-connected (params: number of neurons of the previous layer; number of output neurons)
            nn.ReLU(),
            nn.Linear(128,2) # fully-connected (params: number of neurons of the previous layer; number of output neurons - IF IT's THE LAST LAYER THEN IT'S EQUAL TO NUMBER OF CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# create model
model = BinaryModel().to(device)

############################################################################
######                    hiperparameters                             ######
############################################################################

# NUMBER OF EPOCHS TO SELECT: 5, 10, 20, 30, 40 lub 100
epochs = 30 #

# OPTIMIZER (where lr -> learning rate): Adam (lr = 0.001) lub SGD (lr = 0.01)
opt = optim.Adam(model.parameters(), lr=1e-3) 

# LOSS FUNCTION (DON'T CHANGE; but normally it's good to try various)
loss_fn = nn.CrossEntropyLoss()

model_bin = binary_classification_training(model, opt, epochs, loss_fn, device, n_classes, train_bin, val_bin, test_bin, train_loader_bin, val_loader_bin, test_loader_bin, if_train)

---
1.2B Validation: quality metrics

In [ ]:
# JUST RUN
binary_classification_validate(info, epochs, device, model_bin, test_loader_bin, if_train)

---
1.2C Validation: GRAD-CAM

In [ ]:
# JUST RUN
binary_classification_GRADCAM(model_bin, test_bin, device)


`` JUSTIFY YOUR CHOICE: ``

Numbers of epoch = 20/30
Optimazer = SGD
According to numbers of epoch: If there will be more than 30 it goes to overfitting, less than 20 it goes to underfitting.
According to optimazer, in my project I use Adam, but here with SGB curves are smoother.
